# Compare the three recommendation modes

Run notebook 06b to prepare the place and service snapshots before searching.
Enter a service and a starting location. Each mode shows up to five ranked recommendations.
The default example uses fictional facilities and straight-line distance.
Persistent moderation can be enabled with a private database path.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

# Fill this in if the notebook's working directory is outside the repository.
REPO_ROOT = os.environ.get("DATATHON_REPO_ROOT") or None
cwd = Path.cwd().resolve()
ROOT = Path(REPO_ROOT).expanduser().resolve() if REPO_ROOT else next(
    (folder for folder in (cwd, *cwd.parents)
     if (folder / "non-ml" / "config" / "project.json").is_file()),
    None,
)
if ROOT is None or not (ROOT / "non-ml" / "config" / "project.json").is_file():
    raise RuntimeError("Set REPO_ROOT to the directory that contains non-ml/config/project.json.")
sys.path.insert(0, str(ROOT / "non-ml" / "src"))

from facility_tool.workspace import Store, read_config, show

config = read_config(ROOT)
MODE = os.environ.get("DATATHON_MODE", config["input_mode"])
STORE = Store(ROOT, MODE, config["as_of"])
print(f"Input mode: {MODE} | Evidence date: {config['as_of']}")

## One search, three modes
The same documented service rules feed all three modes. `confirmed` uses documented
fields, `ml` reads authorized ML model outputs if available, and `regression` reads
authorized classical model outputs if available. Model-only service scores appear as
leads to verify. The demo estimates were written by hand to exercise the distinction.

Set `CONSIDER_EQUITY_SCORE` to `True` to use equity scores in balanced sorting.

<details>
<summary>What's that?</summary>
Women and people from marginalized groups can receive worse care. The equity score
combines patient-reported respect and clinical outcomes to help choose between similar hospitals.
</details>

In [ ]:
from datetime import date

from facility_tool.equity import load_paired_equity_evidence, synthetic_equity_evidence
from facility_tool.lookup import lookup, side_by_side
from facility_tool.response import LOAD_OPT_IN_LABEL, new_alternatives, record_call, report_unconfirmed_call

CONCERN = config['lookup']['service']  # Example: 'I need dialysis'
LOCATION = config['lookup']['location']  # City, district, PIN or latitude,longitude
# Future checkbox label; the logic stays off unless explicitly enabled.
print(LOAD_OPT_IN_LABEL)
SPREAD_DEMAND = False
USE_TRAVELTIME = config['traveltime']['enabled']  # Set False for an offline distance-only comparison.
TRANSPORT_MODE = config['traveltime']['mode']  # driving, public_transport, walking, cycling
ROUTE_SORT = None  # Default distance, or time for public transport. Can also choose 'time'.
FACILITY_SORT = config['lookup']['facility_sort']  # balanced or closest
TRAVEL_WEIGHT = config['lookup']['travel_weight']
MAX_EXTRA_KM = config['lookup']['max_extra_km']
MAX_EXTRA_MINUTES = config['lookup']['max_extra_minutes']
# Optional provider CSV of fresh same-service availability for a common window.
MEASURED_LOAD_CSV = os.environ.get('DATATHON_MEASURED_LOAD_CSV')
CONSIDER_EQUITY_SCORE = False
EQUITY_EVIDENCE_CSV = None  # Optional reviewed clinical outcome and respect aggregates for real data.
equity_evidence = None
if CONSIDER_EQUITY_SCORE and MODE == 'demo':
    equity_evidence = synthetic_equity_evidence(
        'dialysis', 'fdr:demo-12', 'fdr:demo-24', as_of=date.fromisoformat(config['as_of']),
    )
elif CONSIDER_EQUITY_SCORE and EQUITY_EVIDENCE_CSV:
    equity_evidence = load_paired_equity_evidence(EQUITY_EVIDENCE_CSV)

settings = dict(concern=CONCERN, location=LOCATION, spread_demand=SPREAD_DEMAND,
                traveltime_enabled=USE_TRAVELTIME, travel_mode=TRANSPORT_MODE, route_sort=ROUTE_SORT,
                facility_sort=FACILITY_SORT, travel_weight=TRAVEL_WEIGHT,
                max_extra_km=MAX_EXTRA_KM, max_extra_minutes=MAX_EXTRA_MINUTES,
                measured_load=pd.read_csv(MEASURED_LOAD_CSV) if SPREAD_DEMAND and MEASURED_LOAD_CSV else None,
                consider_equity=CONSIDER_EQUITY_SCORE, equity_evidence=equity_evidence)
result = lookup(STORE, config, **settings)
print(f"{result['service']} | {result['origin'].label} | {result['origin'].basis}")
print(f"Transport: {result['transport_mode']} | Requested sort: {result['route_sort']}")
print(f"Facility preference: {result['facility_sort']}")
print(result['routing']['message'])
if CONSIDER_EQUITY_SCORE:
    print(result['summary'].iloc[0]['equity_ranking_reason'])
if MODE == 'demo':
    print('Fictional facilities, values, and contacts. Do not use them for care decisions.')
show(result['summary'])
show(side_by_side(result['rows']))

## Reasons, contacts and travel
TravelTime checks every branch that passes the service and location rules. The starting
location is the origin for route estimates. The standard API considers trips of up to
four hours. The routing summary shows whether estimates cover the full comparison.
Driving sorts by route distance by default. Select walking, cycling, or public transport
to compare the same search. Public transport sorts by estimated travel time.
If the top driving option lacks a route for the requested transport, one of the five
slots can show that branch with its driving route and a note. If no requested-mode
routes are available, the list can show up to five labeled driving alternatives.
`travel_mode` shows how to reach each branch in the list. Offline comparisons and
fictional demos use straight-line distance. Within the same service evidence tier,
balanced ranking gives limited travel credit for independent service corroboration and
a current external assessment. Compare `travel_only_rank`
with `baseline_rank` before considering opt-in load balancing. These credits are
ranking preferences in travel units. The fit score uses
the accepted service citation, validity, precision of the origin, and proximity.
Equity scores use paired clinical outcomes and patient-reported respect from comparable
service cohorts. When the evidence can distinguish nearby branches with comparable
documented services, scores can adjust their order. Compare `pre_equity_rank` with
`rank` and use `equity_evidence` to see the paired measures and source. For separately
shown driving options, compare `pre_equity_rank` with `driving_rank`. The sample
equity scores in demo mode are fictional. Without comparable evidence the ordinary
order applies, as it does in time-critical searches and closest sorting.
A listed phone can be outdated. Call the branch to verify the requested service
and availability before traveling. Inspect the complete `trace` for excluded candidates.

In [ ]:
summary_columns = ['mode', 'kind', 'rank', 'baseline_rank', 'travel_only_rank', 'name',
                   'pre_equity_rank', 'equity_score', 'equity_score_lower',
                   'equity_score_upper', 'equity_evidence',
                   'distance_km', 'route_distance_km', 'travel_minutes',
                   'travel_mode', 'route_note', 'requested_mode_route_status', 'driving_rank',
                   'travel_source_url', 'ranking_travel_metric', 'ranking_travel_reason',
                   'ranking_policy_reason', 'corroboration_credit', 'assessment_credit',
                   'quality_credit_unit', 'balanced_travel_score', 'quality_assessment_status',
                   'quality_evidence',
                   'patient_rating', 'patient_rating_min', 'patient_rating_max', 'patient_rating_response_count',
                   'patient_rating_source_url', 'patient_reviews_search_url',
                   'patient_ranking_reason', 'patient_experience_credit',
                   'location_source_urls', 'location_review_flags',
                   'fit_score', 'fit_score_basis', 'phone_number',
                   'phone_source_urls', 'call_to_verify', 'call_response', 'excerpt',
                   'verification_status', 'supporting_publishers', 'supporting_urls',
                   'source_url', 'scope', 'validity', 'reason', 'load_status', 'load_available_units',
                   'load_unit', 'load_as_of', 'load_window_start', 'load_window_end', 'load_source_url']
show(result['rows'][summary_columns], limit=25)
route_summary = result['route_status']
if not route_summary.empty:
    counts = route_summary.groupby(['travel_mode', 'travel_status'], dropna=False).size()
    show(counts.rename('candidates').reset_index())
show(result['trace'], limit=15)

## Follow up after a call
A future yes/no button should call `record_call` and rerun the same query with that
feedback. A No removes this branch for the selected service across all three modes
and replenishes the list from the next supported options. A Yes marks the user's
response for this search. A No can also create a report in a private moderation stack,
where moderators see the newest open reports first and record a disposition. Report
creation and review leave the original evidence and training data intact.
In real mode, set a facility ID and response after you call. The demo simulates a No.

In [ ]:
feedback = ()
CALL_FACILITY_ID = None
CALL_CONFIRMED = None
MODERATION_DB = None  # An optional private SQLite path, outside Git
REPORT_REASON = 'service_not_confirmed'  # Or 'poor_fit'
REPORT_SUBMISSION_ID = None  # Supply the same token on retrying one report
eligible = result['rows'].loc[result['rows']['mode'].eq('confirmed') &
                              result['rows']['kind'].eq('recommendation')]
if MODE == 'demo' and not eligible.empty:
    CALL_FACILITY_ID = eligible.iloc[0]['facility_id']
    CALL_CONFIRMED = False
    print('Simulating a No on the first fictional facility.')
if CALL_FACILITY_ID:
    if CALL_FACILITY_ID not in set(result['rows']['facility_id']):
        raise ValueError('Choose a facility ID shown in this search.')
    feedback = record_call(feedback, result['service'], CALL_FACILITY_ID, confirmed=CALL_CONFIRMED)
    revised = lookup(STORE, config, **settings, feedback=feedback)
    added = new_alternatives(result['rows'], revised['rows'])
    print('Excluded for this caller and service:', revised['excluded_by_caller'])
    print(revised['routing']['message'])
    if not CALL_CONFIRMED and added.empty:
        print('The search returned no additional recommendation.')
    show(added[summary_columns] if not added.empty else revised['rows'][summary_columns])

In [ ]:
# Keep persistence separate so alternatives are already available if reporting fails.
if MODERATION_DB and CALL_FACILITY_ID and CALL_CONFIRMED is False:
    from facility_tool.moderation import ModerationStack
    stack = ModerationStack(MODERATION_DB, allowed_services=config['service_aliases'])
    report = report_unconfirmed_call(feedback, result['service'], CALL_FACILITY_ID,
                                     moderation_stack=stack, reason=REPORT_REASON,
                                     submission_id=REPORT_SUBMISSION_ID)
    print(f'Moderation report {report.report_id} | {report.status}')